# Lab 9 - Data Cleaning
Automotive Industry Tracker - IT 2012 Unstructured Data

In [15]:
import sys
import os
sys.path.append(os.path.abspath("../src"))

Missing Value Analysis

In [16]:
from analytics.data_loader import load_from_csv
from cleaning.missing_handler import report_missing, save_missing_report

df = load_from_csv()
report = report_missing(df)
save_missing_report(df)

2026-05-04 15:44:41,839 - INFO - Loaded CSV: /Users/elmedin.karisiksystemverification.com/Documents/private/Automotive-Industry-Tracker/data/processed/analytics/raw_export.csv, shape=(591, 56)
2026-05-04 15:44:41,844 - INFO - Missing value report: 55 columns with missing values
2026-05-04 15:44:41,849 - INFO - Missing value report: 55 columns with missing values
2026-05-04 15:44:41,853 - INFO - Missing report saved: /Users/elmedin.karisiksystemverification.com/Documents/private/Automotive-Industry-Tracker/data/processed/cleaned/missing_report.csv


                           missing_count  missing_pct
data.text                            578        97.80
language_probability                 575        97.29
data.raw_text                        572        96.79
stored_at                            567        95.94
segment_count                        567        95.94
segments                             567        95.94
model                                567        95.94
duration                             567        95.94
language                             567        95.94
source_file                          567        95.94
data.processed_text                  550        93.06
data.paragraphs                      542        91.71
data.tables                          542        91.71
data.sheets                          542        91.71
data.pages                           532        90.02
data.exif.date_taken                 531        89.85
data.height                          531        89.85
data.file_size_kb           

'/Users/elmedin.karisiksystemverification.com/Documents/private/Automotive-Industry-Tracker/data/processed/cleaned/missing_report.csv'

String Cleaning

In [17]:
from cleaning.string_cleaner import run_string_cleaning

df_clean = df.copy()
df_clean = run_string_cleaning(df_clean)
print(df_clean[["data.make", "source", "data.model"]].head())

2026-05-04 15:44:42,380 - INFO - Starting string cleaning
2026-05-04 15:44:42,383 - INFO - Cleaned title column: 'data.make'
2026-05-04 15:44:42,385 - INFO - Normalized source column: 'source'
2026-05-04 15:44:42,387 - INFO - Normalized model column: 'data.model'
2026-05-04 15:44:42,390 - INFO - Cleaned text column: 'data.description'
2026-05-04 15:44:42,393 - INFO - Extracted year from 'fetched_at' into 'release_year'
2026-05-04 15:44:42,394 - INFO - String cleaning complete


    data.make     source data.model
0        Audi       test         A4
1       Skoda  nhtsa_api    OCTAVIA
2  Volkswagen  nhtsa_api       GOLF
3        Audi  nhtsa_api         A4
4  Volkswagen  nhtsa_api     PASSAT


Deduplication

In [18]:
from cleaning.deduplicator import count_duplicates, drop_exact_duplicates, drop_duplicate_ids

print("Rows before:", len(df_clean))
count_duplicates(df_clean)
df_clean = drop_exact_duplicates(df_clean)
count_duplicates(df_clean, col="source")
df_clean = drop_duplicate_ids(df_clean, id_col="source")
print("Rows after:", len(df_clean))

2026-05-04 15:44:42,797 - INFO - Exact duplicate rows: 0
2026-05-04 15:44:42,804 - INFO - Dropped 0 exact duplicate rows
2026-05-04 15:44:42,805 - INFO - Duplicate values in 'source': 579
2026-05-04 15:44:42,807 - INFO - Dropped 579 rows with duplicate 'source'


Rows before: 591
Exact duplicate rows: 0
Rows before: 591, after dropping exact duplicates: 591
Duplicate values in 'source': 579
Rows after dropping duplicate 'source': 12
Rows after: 12


Type Conversion

In [19]:
from cleaning.type_converter import run_type_conversion

df_before = df_clean.copy()
df_clean = run_type_conversion(df_clean)
print(df_clean.dtypes)

2026-05-04 15:44:43,176 - INFO - Starting type conversion
2026-05-04 15:44:43,180 - INFO - Converted 'fetched_at' to datetime
2026-05-04 15:44:43,182 - INFO - Converted 'data.scraped_at' to datetime
2026-05-04 15:44:43,184 - INFO - Converted 'data.processed_at' to datetime
2026-05-04 15:44:43,186 - INFO - Converted 'stored_at' to datetime
2026-05-04 15:44:43,187 - INFO - Converted 'version' to float32
2026-05-04 15:44:43,188 - INFO - Converted 'data.file_size_kb' to float32
2026-05-04 15:44:43,190 - INFO - Converted 'data.width' to float32
2026-05-04 15:44:43,191 - INFO - Converted 'data.height' to float32
2026-05-04 15:44:43,192 - INFO - Converted 'source' to category
2026-05-04 15:44:43,193 - INFO - Converted '_collection' to category
2026-05-04 15:44:43,195 - INFO - Converted 'data.make' to category
2026-05-04 15:44:43,196 - INFO - Converted 'data.model' to category
2026-05-04 15:44:43,197 - INFO - Converted 'data.type' to category
2026-05-04 15:44:43,201 - INFO - Memory report - be

Memory before : 0.04 MB
Memory after  : 0.04 MB
Reduction     : 0.00 MB
source                             category
fetched_at                   datetime64[ns]
version                             float32
_collection                        category
data.make                          category
data.model                         category
data.year                           float64
data.recalls                         object
data.file_name                       object
data.document_type                   object
data.source                          object
data.extraction_timestamp            object
data.extraction_library              object
data.pages                           object
data.paragraphs                      object
data.tables                          object
data.sheets                          object
data.scraped_at              datetime64[ns]
data.type                          category
data.raw_text                        object
data.processed_text                  object
data

Validation

In [20]:
from cleaning.validator import run_validation

run_validation(df_clean)

2026-05-04 15:44:43,537 - INFO - Starting validation
2026-05-04 15:44:43,539 - INFO - Validation passed: 'source' has no null values
2026-05-04 15:44:43,540 - INFO - Validation passed: '_collection' has no null values
2026-05-04 15:44:43,542 - INFO - Validation passed: all years in 'release_year' are within [1900, 2030]
2026-05-04 15:44:43,543 - INFO - Validation passed: 'fetched_at' dtype is datetime64[ns]
2026-05-04 15:44:43,544 - INFO - Validation passed: 'version' dtype is float32
2026-05-04 15:44:43,545 - INFO - All validations passed


Validation passed: no null values in critical columns
Validation passed: year range [1900, 2030]
Validation passed: all column types correct
All validations passed


Full Cleaning Pipeline

In [21]:
from cleaning.clean_pipeline import run_cleaning_pipeline

df_raw = load_from_csv()
df_final = run_cleaning_pipeline(df_raw)
print(df_final.shape)

2026-05-04 15:44:43,937 - INFO - Loaded CSV: /Users/elmedin.karisiksystemverification.com/Documents/private/Automotive-Industry-Tracker/data/processed/analytics/raw_export.csv, shape=(591, 56)
2026-05-04 15:44:43,938 - INFO - Starting cleaning pipeline
2026-05-04 15:44:43,939 - INFO - Step 1: Dropping high missing columns
2026-05-04 15:44:43,945 - INFO - Dropped 15 columns with >90.0% missing: ['data.pages', 'data.paragraphs', 'data.tables', 'data.sheets', 'data.raw_text', 'data.processed_text', 'data.text', 'source_file', 'language', 'language_probability', 'duration', 'model', 'segments', 'segment_count', 'stored_at']
2026-05-04 15:44:43,946 - INFO - Step 2: Dropping rows with missing critical columns
2026-05-04 15:44:43,948 - INFO - Dropped 24 rows with missing critical columns: ['source', '_collection']
2026-05-04 15:44:43,948 - INFO - Step 3: Filling text fields
2026-05-04 15:44:43,950 - INFO - Filled missing text in 'data.make' with 'unknown'
2026-05-04 15:44:43,951 - INFO - Fill

Dropped columns: ['data.pages', 'data.paragraphs', 'data.tables', 'data.sheets', 'data.raw_text', 'data.processed_text', 'data.text', 'source_file', 'language', 'language_probability', 'duration', 'model', 'segments', 'segment_count', 'stored_at']
Dropped 24 rows missing critical columns
Rows before deduplication: 567
Exact duplicate rows: 0
Rows before: 567, after dropping exact duplicates: 567
Duplicate values in 'source': 556
Rows after dropping duplicate 'source': 11
Rows after dropping duplicate title+date: 11
Memory before : 0.02 MB
Memory after  : 0.02 MB
Reduction     : 0.00 MB
Validation passed: no null values in critical columns
Validation passed: year range [1900, 2030]
Validation passed: all column types correct
All validations passed
Cleaned dataset saved: /Users/elmedin.karisiksystemverification.com/Documents/private/Automotive-Industry-Tracker/data/processed/cleaned/cleaned_data.csv
Final shape: (11, 42)
(11, 42)
